# 06 - Clustering & Forecasting Review

This notebook uses K-Means to group daily load shapes and keeps the final forecast comparison easy to read.

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

hourly = pd.read_csv("../01_Data/processed/hourly_energy.csv", parse_dates=["datetime"]).set_index("datetime")
hourly["day"] = hourly['date_time'].normalize()
hourly["hour"] = hourly['date_time'].hour
profiles = hourly.pivot_table(index="day", columns="hour", values="energy_kwh", aggfunc="mean").dropna()
profiles = profiles.reindex(columns=range(24))

In [ ]:
scaled = profiles.div(profiles.mean(axis=1), axis=0)
model = KMeans(n_clusters=3, random_state=42, n_init=10)
profiles["cluster"] = model.fit_predict(scaled)
print(profiles["cluster"].value_counts().sort_index())

In [ ]:
centers = pd.DataFrame(model.cluster_centers_, columns=range(24))
for i in range(3):
    plt.plot(centers.loc[i], label=f"Cluster {i}")
plt.xlabel("Hour of day")
plt.ylabel("Normalized load")
plt.legend()
plt.tight_layout()
plt.show()